In [1]:
import pandas as pd
import numpy as np
import math

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
DATA_PATH = "data/processed/clean_products.csv"

df = pd.read_csv(DATA_PATH)
pd.set_option("display.max_columns", None)  
print("Shape:", df.shape)
df.head()

Shape: (1351, 23)


,product_id,product_name,category_path,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,img_link,product_link,cat_level_1,cat_level_2,cat_level_3,cat_level_4,cat_level_5,cat_level_6,cat_level_7,product_name_clean,about_product_clean,category_text,clean_text_no_category,clean_text_with_category
0,B002PD61Y4,D-Link DWA-131 300 Mbps Wireless Nano USB Adap...,Computers&Accessories|NetworkingDevices|Networ...,507.0,1208.0,58.0,4.1,8131.0,Connects your computer to a high-speed wireles...,https://m.media-amazon.com/images/I/31+NwZ8gb1...,https://www.amazon.in/D-Link-DWA-131-Wireless-...,Computers&Accessories,NetworkingDevices,NetworkAdapters,WirelessUSBAdapters,Unknown,Unknown,Unknown,d link dwa 131 300 mbps wireless nano usb adap...,connects your computer to a high speed wireles...,computers and accessories networkingdevices ne...,d link dwa 131 300 mbps wireless nano usb adap...,d link dwa 131 300 mbps wireless nano usb adap...
1,B002SZEOLG,TP-Link Nano USB WiFi Dongle 150Mbps High Gain...,Computers&Accessories|NetworkingDevices|Networ...,749.0,1339.0,44.0,4.2,179692.0,150 Mbps Wi-Fi —— Exceptional wireless speed u...,https://m.media-amazon.com/images/I/31Wb+A3VVd...,https://www.amazon.in/TP-Link-TL-WN722N-150Mbp...,Computers&Accessories,NetworkingDevices,NetworkAdapters,WirelessUSBAdapters,Unknown,Unknown,Unknown,tp link nano usb wifi dongle 150mbps high gain...,150 mbps wi fi exceptional wireless speed up t...,computers and accessories networkingdevices ne...,tp link nano usb wifi dongle 150mbps high gain...,tp link nano usb wifi dongle 150mbps high gain...
2,B003B00484,Duracell Plus AAA Rechargeable Batteries (750 ...,Electronics|GeneralPurposeBatteries&BatteryCha...,399.0,499.0,20.0,4.3,27201.0,Duracell Rechargeable AAA 750mAh batteries sta...,https://m.media-amazon.com/images/I/418YrbHVLC...,https://www.amazon.in/Duracell-AAA-750mAh-Rech...,Electronics,GeneralPurposeBatteries&BatteryChargers,RechargeableBatteries,Unknown,Unknown,Unknown,Unknown,duracell plus aaa rechargeable batteries 750 m...,duracell rechargeable aaa 750mah batteries sta...,electronics generalpurposebatteries and batter...,duracell plus aaa rechargeable batteries 750 m...,duracell plus aaa rechargeable batteries 750 m...
3,B003L62T7W,"Logitech B100 Wired USB Mouse, 3 yr Warranty, ...",Computers&Accessories|Accessories&Peripherals|...,279.0,375.0,26.0,4.3,31534.0,"A comfortable, ambidextrous shape feels good i...",https://m.media-amazon.com/images/I/31iFF1Kbkp...,https://www.amazon.in/Logitech-B100-Optical-Mo...,Computers&Accessories,Accessories&Peripherals,"Keyboards,Mice&InputDevices",Mice,Unknown,Unknown,Unknown,logitech b100 wired usb mouse 3 yr warranty 80...,a comfortable ambidextrous shape feels good in...,computers and accessories accessories and peri...,logitech b100 wired usb mouse 3 yr warranty 80...,logitech b100 wired usb mouse 3 yr warranty 80...
4,B004IO5BMQ,"Logitech M235 Wireless Mouse, 1000 DPI Optical...",Computers&Accessories|Accessories&Peripherals|...,699.0,995.0,30.0,4.5,54405.0,You can surf the Web with more comfort and eas...,https://m.media-amazon.com/images/I/31CtVvtFt+...,https://www.amazon.in/Logitech-M235-Wireless-M...,Computers&Accessories,Accessories&Peripherals,"Keyboards,Mice&InputDevices",Mice,Unknown,Unknown,Unknown,logitech m235 wireless mouse 1000 dpi optical ...,you can surf the web with more comfort and eas...,computers and accessories accessories and peri...,logitech m235 wireless mouse 1000 dpi optical ...,logitech m235 wireless mouse 1000 dpi optical ...


In [3]:
df.columns.tolist()

['product_id',
 'product_name',
 'category_path',
 'discounted_price',
 'actual_price',
 'discount_percentage',
 'rating',
 'rating_count',
 'about_product',
 'img_link',
 'product_link',
 'cat_level_1',
 'cat_level_2',
 'cat_level_3',
 'cat_level_4',
 'cat_level_5',
 'cat_level_6',
 'cat_level_7',
 'product_name_clean',
 'about_product_clean',
 'category_text',
 'clean_text_no_category',
 'clean_text_with_category']

In [4]:
def clean_text_value(x):
    if pd.isna(x):
        return ""
    
    x = str(x).strip()
    
    if x.lower() in ["", "nan", "none", "null", "undefined"]:
        return ""
    
    return x


def parse_rating(x):
    try:
        if pd.isna(x):
            return 0.0
        
        x = str(x).strip()
        
        if x.lower() in ["", "nan", "none", "null", "undefined"]:
            return 0.0
        
        return float(x)
    except:
        return 0.0


def parse_rating_count(x):
    try:
        if pd.isna(x):
            return 0
        
        x = str(x).replace(",", "").strip()
        
        if x.lower() in ["", "nan", "none", "null", "undefined"]:
            return 0
        
        return int(float(x))
    except:
        return 0


df["rating_clean"] = df["rating"].apply(parse_rating)
df["rating_count_clean"] = df["rating_count"].apply(parse_rating_count)

df["cat_level_3_clean"] = df["cat_level_3"].apply(clean_text_value)
df["cat_level_4_clean"] = df["cat_level_4"].apply(clean_text_value)


def get_eval_category(row):
    """
    Ưu tiên cat_level_4.
    Nếu không có cat_level_4 thì dùng cat_level_3.
    """
    if row["cat_level_4_clean"] != "":
        return row["cat_level_4_clean"]
    
    if row["cat_level_3_clean"] != "":
        return row["cat_level_3_clean"]
    
    return "Unknown"


df["eval_category"] = df.apply(get_eval_category, axis=1)

df[[
    "product_name",
    "cat_level_3_clean",
    "cat_level_4_clean",
    "eval_category",
    "rating_clean",
    "rating_count_clean"
]].head()

,product_name,cat_level_3_clean,cat_level_4_clean,eval_category,rating_clean,rating_count_clean
0,D-Link DWA-131 300 Mbps Wireless Nano USB Adap...,NetworkAdapters,WirelessUSBAdapters,WirelessUSBAdapters,4.1,8131
1,TP-Link Nano USB WiFi Dongle 150Mbps High Gain...,NetworkAdapters,WirelessUSBAdapters,WirelessUSBAdapters,4.2,179692
2,Duracell Plus AAA Rechargeable Batteries (750 ...,RechargeableBatteries,Unknown,Unknown,4.3,27201
3,"Logitech B100 Wired USB Mouse, 3 yr Warranty, ...","Keyboards,Mice&InputDevices",Mice,Mice,4.3,31534
4,"Logitech M235 Wireless Mouse, 1000 DPI Optical...","Keyboards,Mice&InputDevices",Mice,Mice,4.5,54405


In [5]:
if "clean_text_no_category" not in df.columns:
    df["clean_text_no_category"] = (
        df["product_name"].fillna("").astype(str) + " " +
        df["about_product"].fillna("").astype(str)
    )

if "clean_text_with_category" not in df.columns:
    df["clean_text_with_category"] = (
        df["product_name"].fillna("").astype(str) + " " +
        df["about_product"].fillna("").astype(str) + " " +
        df["cat_level_3_clean"].fillna("").astype(str) + " " +
        df["cat_level_4_clean"].fillna("").astype(str)
    )

df["clean_text_no_category"] = df["clean_text_no_category"].fillna("").astype(str)
df["clean_text_with_category"] = df["clean_text_with_category"].fillna("").astype(str)

df[["clean_text_no_category", "clean_text_with_category"]].head()

,clean_text_no_category,clean_text_with_category
0,d link dwa 131 300 mbps wireless nano usb adap...,d link dwa 131 300 mbps wireless nano usb adap...
1,tp link nano usb wifi dongle 150mbps high gain...,tp link nano usb wifi dongle 150mbps high gain...
2,duracell plus aaa rechargeable batteries 750 m...,duracell plus aaa rechargeable batteries 750 m...
3,logitech b100 wired usb mouse 3 yr warranty 80...,logitech b100 wired usb mouse 3 yr warranty 80...
4,logitech m235 wireless mouse 1000 dpi optical ...,logitech m235 wireless mouse 1000 dpi optical ...


In [6]:
category_to_indices = {}

for idx, cat in enumerate(df["eval_category"]):
    category_to_indices.setdefault(cat, []).append(idx)


def get_relevant_items(query_idx):
    """
    Ground truth giả lập theo category.
    Relevant items là các sản phẩm khác cùng eval_category.
    """
    cat = df.loc[query_idx, "eval_category"]
    relevant = set(category_to_indices.get(cat, []))
    relevant.discard(query_idx)
    return relevant


# Test thử
sample_idx = 0
print("Product:", df.loc[sample_idx, "product_name"])
print("Category:", df.loc[sample_idx, "eval_category"])
print("Number of relevant items:", len(get_relevant_items(sample_idx)))

Product: D-Link DWA-131 300 Mbps Wireless Nano USB Adapter (Black)
Category: WirelessUSBAdapters
Number of relevant items: 13


In [7]:
vectorizer_no_cat = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

vectorizer_with_cat = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

tfidf_no_cat = vectorizer_no_cat.fit_transform(df["clean_text_no_category"])
tfidf_with_cat = vectorizer_with_cat.fit_transform(df["clean_text_with_category"])

print("TF-IDF no category:", tfidf_no_cat.shape)
print("TF-IDF with category:", tfidf_with_cat.shape)

TF-IDF no category: (1351, 5000)
TF-IDF with category: (1351, 5000)


In [8]:
def normalize_rating(rating):
    rating = float(rating)
    return max(0.0, min(rating / 5.0, 1.0))


def normalize_popularity(rating_count):
    """
    Dùng log để giảm độ lệch vì rating_count có thể rất lớn.
    """
    value = math.log1p(int(rating_count))
    
    # log1p(100000) khoảng 11.5, nên chia 12 để scale về gần [0,1]
    return max(0.0, min(value / 12.0, 1.0))


rating_scores = df["rating_clean"].apply(normalize_rating).values
popularity_scores = df["rating_count_clean"].apply(normalize_popularity).values

print(rating_scores[:5])
print(popularity_scores[:5])

[0.82 0.84 0.86 0.86 0.9 ]
[0.75029685 1.         0.85092048 0.86323778 0.90868581]


In [9]:
def recommend_content_only(query_idx, top_k=10):
    """
    Model 1:
    Content only = title + description
    Không dùng category.
    """
    sim_scores = cosine_similarity(tfidf_no_cat[query_idx], tfidf_no_cat).flatten()
    sim_scores[query_idx] = -1
    
    ranked_indices = np.argsort(sim_scores)[::-1][:top_k]
    return ranked_indices.tolist()


def recommend_content_with_category(query_idx, top_k=10):
    """
    Model 2:
    Content + Category = title + description + category
    """
    sim_scores = cosine_similarity(tfidf_with_cat[query_idx], tfidf_with_cat).flatten()
    sim_scores[query_idx] = -1
    
    ranked_indices = np.argsort(sim_scores)[::-1][:top_k]
    return ranked_indices.tolist()


def recommend_hybrid(query_idx, top_k=10, w_sim=0.75, w_rating=0.15, w_pop=0.10):
    """
    Model 3:
    Hybrid = content + category + rating + popularity
    """
    sim_scores = cosine_similarity(tfidf_with_cat[query_idx], tfidf_with_cat).flatten()
    sim_scores[query_idx] = -1
    
    final_scores = (
        w_sim * sim_scores +
        w_rating * rating_scores +
        w_pop * popularity_scores
    )
    
    final_scores[query_idx] = -1
    
    ranked_indices = np.argsort(final_scores)[::-1][:top_k]
    return ranked_indices.tolist()

In [10]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    
    if k == 0:
        return 0.0
    
    hit_count = len(set(recommended_k) & relevant)
    return hit_count / k


def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    return 1.0 if len(set(recommended_k) & relevant) > 0 else 0.0


def ndcg_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    
    dcg = 0.0
    
    for i, item_idx in enumerate(recommended_k):
        if item_idx in relevant:
            dcg += 1.0 / math.log2(i + 2)
    
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_hits))
    
    if idcg == 0:
        return 0.0
    
    return dcg / idcg


def avg_rating_at_k(recommended, k):
    recommended_k = recommended[:k]
    
    if len(recommended_k) == 0:
        return 0.0
    
    return df.loc[recommended_k, "rating_clean"].mean()


def avg_popularity_at_k(recommended, k):
    recommended_k = recommended[:k]
    
    if len(recommended_k) == 0:
        return 0.0
    
    return np.log1p(df.loc[recommended_k, "rating_count_clean"]).mean()

In [11]:
valid_query_indices = []

for idx in range(len(df)):
    relevant = get_relevant_items(idx)
    
    if len(relevant) >= 3:
        valid_query_indices.append(idx)

print("Số sản phẩm có thể đánh giá:", len(valid_query_indices))

np.random.seed(42)

sample_size = min(300, len(valid_query_indices))
eval_indices = np.random.choice(valid_query_indices, size=sample_size, replace=False)

print("Số sản phẩm dùng để đánh giá:", len(eval_indices))

Số sản phẩm có thể đánh giá: 1225
Số sản phẩm dùng để đánh giá: 300


In [12]:
def evaluate_model(model_name, recommend_func, k=10):
    rows = []
    
    for query_idx in eval_indices:
        relevant = get_relevant_items(query_idx)
        recommended = recommend_func(query_idx, top_k=k)
        
        rows.append({
            "Model": model_name,
            f"Precision@{k}": precision_at_k(recommended, relevant, k),
            f"NDCG@{k}": ndcg_at_k(recommended, relevant, k),
            f"HitRate@{k}": hit_rate_at_k(recommended, relevant, k),
            f"AvgRating@{k}": avg_rating_at_k(recommended, k),
            f"AvgPopularity@{k}": avg_popularity_at_k(recommended, k),
        })
    
    result = pd.DataFrame(rows).groupby("Model").mean().reset_index()
    return result

In [13]:
K = 10

result_content_only = evaluate_model(
    "Content Only",
    recommend_content_only,
    k=K
)

result_content_category = evaluate_model(
    "Content + Category",
    recommend_content_with_category,
    k=K
)

result_hybrid = evaluate_model(
    "Hybrid",
    lambda query_idx, top_k: recommend_hybrid(
        query_idx,
        top_k=top_k,
        w_sim=0.75,
        w_rating=0.15,
        w_pop=0.10
    ),
    k=K
)

metrics_table = pd.concat(
    [
        result_content_only,
        result_content_category,
        result_hybrid
    ],
    ignore_index=True
)

metrics_table

,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10
0,Content Only,0.771333,0.836973,0.983333,4.078400,8.353494
1,Content + Category,0.813667,0.875619,0.990000,4.075133,8.352294
2,Hybrid,0.807667,0.872118,0.990000,4.114867,8.724431


In [14]:
metrics_table_rounded = metrics_table.copy()

for col in metrics_table_rounded.columns:
    if col != "Model":
        metrics_table_rounded[col] = metrics_table_rounded[col].round(4)

metrics_table_rounded

,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10
0,Content Only,0.7713,0.8370,0.9833,4.0784,8.3535
1,Content + Category,0.8137,0.8756,0.9900,4.0751,8.3523
2,Hybrid,0.8077,0.8721,0.9900,4.1149,8.7244


In [15]:
hybrid_configs = [
    ("Hybrid 0.80/0.10/0.10", 0.80, 0.10, 0.10),
    ("Hybrid 0.75/0.15/0.10", 0.75, 0.15, 0.10),
    ("Hybrid 0.70/0.15/0.15", 0.70, 0.15, 0.15),
]

hybrid_results = []

for name, w_sim, w_rating, w_pop in hybrid_configs:
    result = evaluate_model(
        name,
        lambda query_idx, top_k, w_sim=w_sim, w_rating=w_rating, w_pop=w_pop: recommend_hybrid(
            query_idx,
            top_k=top_k,
            w_sim=w_sim,
            w_rating=w_rating,
            w_pop=w_pop
        ),
        k=K
    )
    hybrid_results.append(result)

hybrid_metrics_table = pd.concat(hybrid_results, ignore_index=True)

for col in hybrid_metrics_table.columns:
    if col != "Model":
        hybrid_metrics_table[col] = hybrid_metrics_table[col].round(4)

hybrid_metrics_table

,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10
0,Hybrid 0.80/0.10/0.10,0.8083,0.8733,0.99,4.1049,8.6874
1,Hybrid 0.75/0.15/0.10,0.8077,0.8721,0.99,4.1149,8.7244
2,Hybrid 0.70/0.15/0.15,0.8057,0.8709,0.99,4.1214,8.8989


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

tfidf_param_grid = [
    {
        "max_features": 3000,
        "ngram_range": (1, 1),
        "min_df": 1
    },
    {
        "max_features": 5000,
        "ngram_range": (1, 1),
        "min_df": 1
    },
    {
        "max_features": 5000,
        "ngram_range": (1, 2),
        "min_df": 1
    },
    {
        "max_features": 8000,
        "ngram_range": (1, 1),
        "min_df": 1
    },
    {
        "max_features": 8000,
        "ngram_range": (1, 2),
        "min_df": 1
    },
    {
        "max_features": 8000,
        "ngram_range": (1, 2),
        "min_df": 2
    }
]


def evaluate_tfidf_content_only(config, k=10):
    vectorizer = TfidfVectorizer(
        max_features=config["max_features"],
        ngram_range=config["ngram_range"],
        min_df=config["min_df"],
        stop_words="english"
    )

    tfidf_matrix = vectorizer.fit_transform(df["clean_text_no_category"])

    def recommend_func(query_idx, top_k=10):
        sim_scores = cosine_similarity(
            tfidf_matrix[query_idx],
            tfidf_matrix
        ).flatten()

        sim_scores[query_idx] = -1

        ranked_indices = np.argsort(sim_scores)[::-1][:top_k]
        return ranked_indices.tolist()

    result = evaluate_model(
        model_name=(
            f"Content Only | "
            f"max_features={config['max_features']}, "
            f"ngram={config['ngram_range']}, "
            f"min_df={config['min_df']}"
        ),
        recommend_func=recommend_func,
        k=k
    )

    result["max_features"] = config["max_features"]
    result["ngram_range"] = str(config["ngram_range"])
    result["min_df"] = config["min_df"]

    return result


tfidf_content_only_results = []

for config in tfidf_param_grid:
    result = evaluate_tfidf_content_only(config, k=10)
    tfidf_content_only_results.append(result)

tfidf_content_only_table = pd.concat(
    tfidf_content_only_results,
    ignore_index=True
)

for col in tfidf_content_only_table.columns:
    if col not in ["Model", "ngram_range"]:
        tfidf_content_only_table[col] = tfidf_content_only_table[col].round(4)

tfidf_content_only_table

,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10,max_features,ngram_range,min_df
0,"Content Only | max_features=3000, ngram=(1, 1)...",0.7763,0.8425,0.9867,4.0780,8.3829,3000,"(1, 1)",1
1,"Content Only | max_features=5000, ngram=(1, 1)...",0.7740,0.8394,0.9833,4.0774,8.3981,5000,"(1, 1)",1
2,"Content Only | max_features=5000, ngram=(1, 2)...",0.7713,0.8370,0.9833,4.0784,8.3535,5000,"(1, 2)",1
3,"Content Only | max_features=8000, ngram=(1, 1)...",0.7723,0.8378,0.9833,4.0786,8.4062,8000,"(1, 1)",1
4,"Content Only | max_features=8000, ngram=(1, 2)...",0.7677,0.8330,0.9833,4.0772,8.3275,8000,"(1, 2)",1
5,"Content Only | max_features=8000, ngram=(1, 2)...",0.7700,0.8352,0.9833,4.0796,8.3374,8000,"(1, 2)",2


In [17]:
def evaluate_tfidf_with_category(config, k=10):
    vectorizer = TfidfVectorizer(
        max_features=config["max_features"],
        ngram_range=config["ngram_range"],
        min_df=config["min_df"],
        stop_words="english"
    )

    tfidf_matrix = vectorizer.fit_transform(df["clean_text_with_category"])

    def recommend_func(query_idx, top_k=10):
        sim_scores = cosine_similarity(
            tfidf_matrix[query_idx],
            tfidf_matrix
        ).flatten()

        sim_scores[query_idx] = -1

        ranked_indices = np.argsort(sim_scores)[::-1][:top_k]
        return ranked_indices.tolist()

    result = evaluate_model(
        model_name=(
            f"Content + Category | "
            f"max_features={config['max_features']}, "
            f"ngram={config['ngram_range']}, "
            f"min_df={config['min_df']}"
        ),
        recommend_func=recommend_func,
        k=k
    )

    result["max_features"] = config["max_features"]
    result["ngram_range"] = str(config["ngram_range"])
    result["min_df"] = config["min_df"]

    return result


tfidf_with_category_results = []

for config in tfidf_param_grid:
    result = evaluate_tfidf_with_category(config, k=10)
    tfidf_with_category_results.append(result)

tfidf_with_category_table = pd.concat(
    tfidf_with_category_results,
    ignore_index=True
)

for col in tfidf_with_category_table.columns:
    if col not in ["Model", "ngram_range"]:
        tfidf_with_category_table[col] = tfidf_with_category_table[col].round(4)

tfidf_with_category_table

,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10,max_features,ngram_range,min_df
0,"Content + Category | max_features=3000, ngram=...",0.8057,0.8712,0.99,4.0762,8.3625,3000,"(1, 1)",1
1,"Content + Category | max_features=5000, ngram=...",0.8073,0.8720,0.99,4.0788,8.3936,5000,"(1, 1)",1
2,"Content + Category | max_features=5000, ngram=...",0.8137,0.8756,0.99,4.0751,8.3523,5000,"(1, 2)",1
3,"Content + Category | max_features=8000, ngram=...",0.8040,0.8684,0.99,4.0798,8.3959,8000,"(1, 1)",1
4,"Content + Category | max_features=8000, ngram=...",0.8103,0.8750,0.99,4.0768,8.3464,8000,"(1, 2)",1
5,"Content + Category | max_features=8000, ngram=...",0.8110,0.8750,0.99,4.0768,8.3362,8000,"(1, 2)",2


In [18]:
best_content_only = tfidf_content_only_table.sort_values(
    by="NDCG@10",
    ascending=False
).head(1)

best_content_category = tfidf_with_category_table.sort_values(
    by="NDCG@10",
    ascending=False
).head(1)

print("Best Content Only:")
display(best_content_only)

print("Best Content + Category:")
display(best_content_category)

Best Content Only:


,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10,max_features,ngram_range,min_df
0,"Content Only | max_features=3000, ngram=(1, 1)...",0.7763,0.8425,0.9867,4.078,8.3829,3000,"(1, 1)",1


Best Content + Category:


,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10,max_features,ngram_range,min_df
2,"Content + Category | max_features=5000, ngram=...",0.8137,0.8756,0.99,4.0751,8.3523,5000,"(1, 2)",1


In [19]:
tfidf_content_only_table.to_csv(
    "tfidf_content_only_tuning.csv",
    index=False
)

tfidf_with_category_table.to_csv(
    "tfidf_with_category_tuning.csv",
    index=False
)

print("Saved:")
print("- tfidf_content_only_tuning.csv")
print("- tfidf_with_category_tuning.csv")

Saved:
- tfidf_content_only_tuning.csv
- tfidf_with_category_tuning.csv



# PHẦN MỞ RỘNG: So sánh TF-IDF Content + Category, Weighted TF-IDF và Sentence-BERT

**Mục tiêu của phần này:** mở rộng notebook đánh giá hiện tại bằng cách chỉ so sánh các phương pháp dựa trên `content + category`, gồm:

1. **TF-IDF Content + Category**: baseline hiện tại của bài.
2. **Weighted TF-IDF**: tách từng nhóm thông tin sản phẩm thành `product_name`, `about_product`, `category`, sau đó gán trọng số khác nhau cho từng nhóm.
3. **Sentence-BERT**: dùng embedding ngữ nghĩa để đo độ tương tự giữa các sản phẩm.

**Lưu ý:** Ground truth trong notebook này vẫn là ground truth giả lập theo `eval_category`, tức là sản phẩm được xem là relevant nếu cùng category đánh giá với sản phẩm gốc.



## 1. Chuẩn bị text field cho phần so sánh mở rộng

Cell này tạo 3 nhóm text riêng:

- `text_product_name`: tên sản phẩm, thường rất quan trọng vì chứa ý chính của sản phẩm.
- `text_about_product`: mô tả sản phẩm, chứa nhiều thông tin nhưng cũng dễ có nhiễu.
- `text_category`: category path hoặc các cấp category, dùng để giữ ngữ cảnh danh mục.

Việc tách riêng các field giúp Weighted TF-IDF có thể điều chỉnh mức ảnh hưởng của từng loại thông tin.


In [20]:

# ============================================================
# PHẦN 1: Chuẩn bị dữ liệu text cho Weighted TF-IDF và Sentence-BERT
# ============================================================

# Tên sản phẩm
if "product_name" in df.columns:
    df["text_product_name"] = df["product_name"].apply(clean_text_value)
else:
    df["text_product_name"] = ""

# Mô tả sản phẩm
if "about_product" in df.columns:
    df["text_about_product"] = df["about_product"].apply(clean_text_value)
else:
    df["text_about_product"] = ""

# Category: ưu tiên category_path nếu có, nếu không thì ghép cat_level_3 + cat_level_4
if "category_path" in df.columns:
    df["text_category"] = df["category_path"].apply(clean_text_value)
else:
    df["text_category"] = (
        df["cat_level_3_clean"].fillna("").astype(str) + " " +
        df["cat_level_4_clean"].fillna("").astype(str)
    ).str.strip()

# Text đầy đủ tương ứng với Content + Category baseline
# Dùng lại clean_text_with_category nếu đã có sẵn trong pipeline của bạn.
df["text_content_category"] = df["clean_text_with_category"].fillna("").astype(str)

print("Sample text fields:")
display(df[[
    "product_name",
    "text_product_name",
    "text_about_product",
    "text_category",
    "eval_category"
]].head(3))


Sample text fields:


,product_name,text_product_name,text_about_product,text_category,eval_category
0,D-Link DWA-131 300 Mbps Wireless Nano USB Adap...,D-Link DWA-131 300 Mbps Wireless Nano USB Adap...,Connects your computer to a high-speed wireles...,Computers&Accessories|NetworkingDevices|Networ...,WirelessUSBAdapters
1,TP-Link Nano USB WiFi Dongle 150Mbps High Gain...,TP-Link Nano USB WiFi Dongle 150Mbps High Gain...,150 Mbps Wi-Fi —— Exceptional wireless speed u...,Computers&Accessories|NetworkingDevices|Networ...,WirelessUSBAdapters
2,Duracell Plus AAA Rechargeable Batteries (750 ...,Duracell Plus AAA Rechargeable Batteries (750 ...,Duracell Rechargeable AAA 750mAh batteries sta...,Electronics|GeneralPurposeBatteries&BatteryCha...,Unknown



## 2. Weighted TF-IDF

**Ý tưởng:** TF-IDF Content + Category hiện tại ghép toàn bộ text lại thành một chuỗi chung. Cách này đơn giản nhưng chưa phân biệt được field nào quan trọng hơn.

Weighted TF-IDF trong phần này sẽ tính similarity riêng cho từng nhóm:

\[
score = w_{name} \times sim(name) + w_{about} \times sim(about) + w_{category} \times sim(category)
\]

Trong đó:

- `w_name`: trọng số cho tên sản phẩm.
- `w_about`: trọng số cho mô tả sản phẩm.
- `w_category`: trọng số cho category.

Nếu muốn hệ thống gợi ý bám sát danh mục hơn, tăng `w_category`. Nếu muốn bám sát tên sản phẩm hơn, tăng `w_name`.


In [21]:

# ============================================================
# PHẦN 2: Hàm đánh giá Weighted TF-IDF
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def fit_weighted_tfidf_matrices(tfidf_config):
    """
    Fit 3 TF-IDF matrix riêng cho:
    1. product_name
    2. about_product
    3. category

    tfidf_config là bộ tham số của TfidfVectorizer.
    """
    vectorizer_name = TfidfVectorizer(
        max_features=tfidf_config["max_features"],
        ngram_range=tfidf_config["ngram_range"],
        min_df=tfidf_config["min_df"],
        stop_words="english"
    )

    vectorizer_about = TfidfVectorizer(
        max_features=tfidf_config["max_features"],
        ngram_range=tfidf_config["ngram_range"],
        min_df=tfidf_config["min_df"],
        stop_words="english"
    )

    vectorizer_category = TfidfVectorizer(
        max_features=tfidf_config["max_features"],
        ngram_range=tfidf_config["ngram_range"],
        min_df=tfidf_config["min_df"],
        stop_words="english"
    )

    name_matrix = vectorizer_name.fit_transform(df["text_product_name"])
    about_matrix = vectorizer_about.fit_transform(df["text_about_product"])
    category_matrix = vectorizer_category.fit_transform(df["text_category"])

    return {
        "name_matrix": name_matrix,
        "about_matrix": about_matrix,
        "category_matrix": category_matrix,
        "vectorizer_name": vectorizer_name,
        "vectorizer_about": vectorizer_about,
        "vectorizer_category": vectorizer_category,
    }


def recommend_weighted_tfidf(
    query_idx,
    matrices,
    top_k=10,
    w_name=0.45,
    w_about=0.25,
    w_category=0.30
):
    """
    Recommend bằng Weighted TF-IDF.

    Thay vì ghép toàn bộ text vào một TF-IDF matrix,
    hàm này tính cosine similarity riêng trên từng field rồi cộng theo trọng số.
    """
    # Chuẩn hóa trọng số để tổng = 1, tránh trường hợp nhập trọng số lệch scale.
    weight_sum = w_name + w_about + w_category
    if weight_sum == 0:
        w_name, w_about, w_category = 0.45, 0.25, 0.30
        weight_sum = 1.0

    w_name = w_name / weight_sum
    w_about = w_about / weight_sum
    w_category = w_category / weight_sum

    sim_name = cosine_similarity(
        matrices["name_matrix"][query_idx],
        matrices["name_matrix"]
    ).flatten()

    sim_about = cosine_similarity(
        matrices["about_matrix"][query_idx],
        matrices["about_matrix"]
    ).flatten()

    sim_category = cosine_similarity(
        matrices["category_matrix"][query_idx],
        matrices["category_matrix"]
    ).flatten()

    final_scores = (
        w_name * sim_name +
        w_about * sim_about +
        w_category * sim_category
    )

    final_scores[query_idx] = -1
    ranked_indices = np.argsort(final_scores)[::-1][:top_k]
    return ranked_indices.tolist()


def evaluate_weighted_tfidf_config(tfidf_config, weight_config, k=10):
    """
    Đánh giá một cấu hình Weighted TF-IDF cụ thể.
    Bao gồm:
    - tham số TF-IDF: max_features, ngram_range, min_df
    - trọng số: w_name, w_about, w_category
    """
    matrices = fit_weighted_tfidf_matrices(tfidf_config)

    w_name = weight_config["w_name"]
    w_about = weight_config["w_about"]
    w_category = weight_config["w_category"]

    model_name = (
        "Weighted TF-IDF | "
        f"max_features={tfidf_config['max_features']}, "
        f"ngram={tfidf_config['ngram_range']}, "
        f"min_df={tfidf_config['min_df']}, "
        f"w_name={w_name}, "
        f"w_about={w_about}, "
        f"w_category={w_category}"
    )

    result = evaluate_model(
        model_name=model_name,
        recommend_func=lambda query_idx, top_k: recommend_weighted_tfidf(
            query_idx=query_idx,
            matrices=matrices,
            top_k=top_k,
            w_name=w_name,
            w_about=w_about,
            w_category=w_category
        ),
        k=k
    )

    result["max_features"] = tfidf_config["max_features"]
    result["ngram_range"] = str(tfidf_config["ngram_range"])
    result["min_df"] = tfidf_config["min_df"]
    result["w_name"] = w_name
    result["w_about"] = w_about
    result["w_category"] = w_category

    return result



## 3. Tinh chỉnh tham số cho Weighted TF-IDF

Cell này thử nhiều cấu hình khác nhau:

**Tham số TF-IDF:**

- `max_features`: số lượng từ/cụm từ tối đa được giữ lại.
- `ngram_range`: dùng unigram `(1,1)` hoặc thêm bigram `(1,2)`.
- `min_df`: bỏ các từ xuất hiện quá ít.

**Tham số trọng số:**

- `w_name`: mức ưu tiên tên sản phẩm.
- `w_about`: mức ưu tiên mô tả sản phẩm.
- `w_category`: mức ưu tiên category.

Bạn có thể giải thích trong báo cáo rằng đây là quá trình tìm cấu hình Weighted TF-IDF phù hợp nhất theo `NDCG@10`.


In [22]:

# ============================================================
# PHẦN 3: Tuning Weighted TF-IDF
# ============================================================

weighted_tfidf_param_grid = [
    {"max_features": 5000, "ngram_range": (1, 1), "min_df": 1},
    {"max_features": 5000, "ngram_range": (1, 2), "min_df": 1},
    {"max_features": 8000, "ngram_range": (1, 1), "min_df": 1},
    {"max_features": 8000, "ngram_range": (1, 2), "min_df": 1},
    {"max_features": 8000, "ngram_range": (1, 2), "min_df": 2},
]

weighted_tfidf_weight_grid = [
    # Cân bằng giữa tên, mô tả và category
    {"w_name": 0.40, "w_about": 0.30, "w_category": 0.30},

    # Ưu tiên tên sản phẩm vì tên thường chứa ý chính
    {"w_name": 0.50, "w_about": 0.25, "w_category": 0.25},

    # Ưu tiên category để recommendation bám sát danh mục hơn
    {"w_name": 0.35, "w_about": 0.20, "w_category": 0.45},

    # Ưu tiên tên + category, giảm mô tả vì mô tả dài có thể nhiễu
    {"w_name": 0.45, "w_about": 0.15, "w_category": 0.40},
]

weighted_tfidf_results = []

for tfidf_config in weighted_tfidf_param_grid:
    for weight_config in weighted_tfidf_weight_grid:
        result = evaluate_weighted_tfidf_config(
            tfidf_config=tfidf_config,
            weight_config=weight_config,
            k=K
        )
        weighted_tfidf_results.append(result)

weighted_tfidf_table = pd.concat(weighted_tfidf_results, ignore_index=True)

for col in weighted_tfidf_table.columns:
    if col not in ["Model", "ngram_range"]:
        weighted_tfidf_table[col] = weighted_tfidf_table[col].round(4)

weighted_tfidf_table.sort_values(by=f"NDCG@{K}", ascending=False).head(10)


,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10,max_features,ngram_range,min_df,w_name,w_about,w_category
14,"Weighted TF-IDF | max_features=8000, ngram=(1,...",0.8880,0.9611,0.9933,4.0786,8.3311,8000,"(1, 2)",1,0.35,0.20,0.45
10,"Weighted TF-IDF | max_features=8000, ngram=(1,...",0.8860,0.9604,0.9933,4.0865,8.4000,8000,"(1, 1)",1,0.35,0.20,0.45
2,"Weighted TF-IDF | max_features=5000, ngram=(1,...",0.8860,0.9602,0.9933,4.0861,8.3996,5000,"(1, 1)",1,0.35,0.20,0.45
18,"Weighted TF-IDF | max_features=8000, ngram=(1,...",0.8867,0.9595,0.9933,4.0825,8.3183,8000,"(1, 2)",2,0.35,0.20,0.45
6,"Weighted TF-IDF | max_features=5000, ngram=(1,...",0.8853,0.9580,0.9933,4.0841,8.3405,5000,"(1, 2)",1,0.35,0.20,0.45
3,"Weighted TF-IDF | max_features=5000, ngram=(1,...",0.8847,0.9576,0.9933,4.0886,8.3946,5000,"(1, 1)",1,0.45,0.15,0.40
11,"Weighted TF-IDF | max_features=8000, ngram=(1,...",0.8847,0.9574,0.9933,4.0894,8.3944,8000,"(1, 1)",1,0.45,0.15,0.40
15,"Weighted TF-IDF | max_features=8000, ngram=(1,...",0.8843,0.9572,0.9933,4.0802,8.3514,8000,"(1, 2)",1,0.45,0.15,0.40
19,"Weighted TF-IDF | max_features=8000, ngram=(1,...",0.8820,0.9556,0.9933,4.0859,8.3265,8000,"(1, 2)",2,0.45,0.15,0.40
7,"Weighted TF-IDF | max_features=5000, ngram=(1,...",0.8837,0.9553,0.9933,4.0858,8.3324,5000,"(1, 2)",1,0.45,0.15,0.40


In [23]:

# ============================================================
# PHẦN 4: Lấy cấu hình Weighted TF-IDF tốt nhất
# ============================================================

best_weighted_tfidf = weighted_tfidf_table.sort_values(
    by=f"NDCG@{K}",
    ascending=False
).head(1).copy()

print("Best Weighted TF-IDF config:")
display(best_weighted_tfidf)


Best Weighted TF-IDF config:


,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10,max_features,ngram_range,min_df,w_name,w_about,w_category
14,"Weighted TF-IDF | max_features=8000, ngram=(1,...",0.888,0.9611,0.9933,4.0786,8.3311,8000,"(1, 2)",1,0.35,0.2,0.45



## 4. Sentence-BERT

**Ý tưởng:** TF-IDF và Weighted TF-IDF chủ yếu dựa vào mức độ trùng từ khóa. Sentence-BERT biểu diễn mỗi sản phẩm thành một vector ngữ nghĩa, nên có thể nhận ra các sản phẩm gần nghĩa dù không dùng y hệt từ khóa.

Ví dụ:

- `wireless headphone`
- `bluetooth headset`

Hai cụm này có thể không trùng nhiều từ, nhưng về nghĩa lại khá gần nhau. Đây là lý do Sentence-BERT phù hợp để so sánh với TF-IDF trong bài gợi ý sản phẩm.

**Cần cài thư viện nếu máy bạn chưa có:**

```bash
pip install sentence-transformers
```


In [24]:

# ============================================================
# PHẦN 5: Import Sentence-BERT
# ============================================================

try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
    print("sentence-transformers đã sẵn sàng.")
except ImportError:
    SBERT_AVAILABLE = False
    print("Bạn chưa cài sentence-transformers.")
    print("Hãy chạy lệnh sau trong terminal hoặc notebook:")
    print("pip install sentence-transformers")


d:\DA2TEST\test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence-transformers đã sẵn sàng.



## 5. Tinh chỉnh tham số cho Sentence-BERT

Trong phần này, ta tinh chỉnh theo 2 nhóm chính:

1. **Model Sentence-BERT**
   - `all-MiniLM-L6-v2`: nhẹ, chạy nhanh, phù hợp demo đồ án.
   - `paraphrase-MiniLM-L6-v2`: cũng nhẹ, thường ổn với similarity.
   - `all-mpnet-base-v2`: chất lượng tốt hơn nhưng nặng hơn.

2. **Cách ghép text đầu vào**
   - `name_category_about`: ghép tên + category + mô tả theo cách cân bằng.
   - `name_category_focus`: lặp lại tên và category để nhấn mạnh 2 field quan trọng.
   - `category_name_about`: đưa category lên trước để embedding chú ý danh mục hơn.

`batch_size` chủ yếu ảnh hưởng tốc độ encode, không phải chất lượng model. Tuy nhiên vẫn đưa vào config để bạn dễ kiểm soát khi chạy trên máy yếu hoặc mạnh.


In [25]:

# ============================================================
# PHẦN 6: Hàm tạo text và đánh giá Sentence-BERT
# ============================================================


def build_sbert_text(row, variant="name_category_about"):
    """
    Tạo input text cho Sentence-BERT.

    Mỗi variant thể hiện một cách nhấn mạnh thông tin khác nhau.
    """
    name = clean_text_value(row.get("text_product_name", ""))
    about = clean_text_value(row.get("text_about_product", ""))
    category = clean_text_value(row.get("text_category", ""))

    if variant == "name_category_about":
        return f"Product name: {name}. Category: {category}. Description: {about}"

    if variant == "name_category_focus":
        # Lặp lại name và category để tăng ảnh hưởng của 2 field này trong embedding.
        return f"Product name: {name}. Product name: {name}. Category: {category}. Category: {category}. Description: {about}"

    if variant == "category_name_about":
        # Đưa category lên đầu để nhấn mạnh ngữ cảnh danh mục.
        return f"Category: {category}. Product name: {name}. Description: {about}"

    # Mặc định dùng text Content + Category đang có.
    return clean_text_value(row.get("text_content_category", ""))


def encode_sbert_embeddings(model_name, texts, batch_size=64, normalize_embeddings=True):
    """
    Encode toàn bộ sản phẩm thành embedding bằng Sentence-BERT.

    normalize_embeddings=True giúp cosine similarity có thể tính nhanh bằng dot product.
    """
    model = SentenceTransformer(model_name)

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=normalize_embeddings
    )

    return embeddings


def recommend_from_embeddings(query_idx, embeddings, top_k=10):
    """
    Recommend dựa trên cosine similarity giữa embedding của query product và toàn bộ sản phẩm.

    Vì embeddings đã normalize nên cosine similarity = dot product.
    """
    sim_scores = embeddings[query_idx] @ embeddings.T
    sim_scores[query_idx] = -1

    ranked_indices = np.argsort(sim_scores)[::-1][:top_k]
    return ranked_indices.tolist()


def evaluate_sbert_config(config, k=10):
    """
    Đánh giá một cấu hình Sentence-BERT.
    """
    texts = df.apply(
        lambda row: build_sbert_text(row, variant=config["text_variant"]),
        axis=1
    ).tolist()

    embeddings = encode_sbert_embeddings(
        model_name=config["model_name"],
        texts=texts,
        batch_size=config["batch_size"],
        normalize_embeddings=True
    )

    model_name = (
        "Sentence-BERT | "
        f"model={config['model_name']}, "
        f"text_variant={config['text_variant']}, "
        f"batch_size={config['batch_size']}"
    )

    result = evaluate_model(
        model_name=model_name,
        recommend_func=lambda query_idx, top_k: recommend_from_embeddings(
            query_idx=query_idx,
            embeddings=embeddings,
            top_k=top_k
        ),
        k=k
    )

    result["sbert_model"] = config["model_name"]
    result["text_variant"] = config["text_variant"]
    result["batch_size"] = config["batch_size"]

    return result


In [26]:

# ============================================================
# PHẦN 7: Tuning Sentence-BERT
# ============================================================

# Gợi ý chạy trước 1-2 config nhẹ để kiểm tra.
# Nếu máy chạy chậm, bạn có thể comment bớt model nặng như all-mpnet-base-v2.
sbert_param_grid = [
    # {
    #     "model_name": "all-MiniLM-L6-v2",
    #     "text_variant": "name_category_about",
    #     "batch_size": 64
    # },
    {
        "model_name": "all-MiniLM-L6-v2",
        "text_variant": "name_category_focus",
        "batch_size": 64
    },
    {
        "model_name": "paraphrase-MiniLM-L6-v2",
        "text_variant": "name_category_about",
        "batch_size": 64
    },
    {
        "model_name": "paraphrase-MiniLM-L6-v2",
        "text_variant": "category_name_about",
        "batch_size": 64
    },
    # Model này thường tốt hơn nhưng nặng hơn.
    # Bỏ comment nếu máy đủ mạnh hoặc muốn thử nghiệm thêm.
    # {
    #     "model_name": "all-mpnet-base-v2",
    #     "text_variant": "name_category_about",
    #     "batch_size": 32
    # },
]

sbert_results = []
sbert_errors = []

if SBERT_AVAILABLE:
    for config in sbert_param_grid:
        print("Running Sentence-BERT config:", config)
        try:
            result = evaluate_sbert_config(config, k=K)
            sbert_results.append(result)
        except Exception as e:
            # Nếu model chưa tải được hoặc máy thiếu dependency, lưu lỗi để dễ debug.
            sbert_errors.append({
                "config": config,
                "error": str(e)
            })
            print("Lỗi với config:", config)
            print(e)

    if len(sbert_results) > 0:
        sbert_table = pd.concat(sbert_results, ignore_index=True)

        for col in sbert_table.columns:
            if col not in ["Model", "sbert_model", "text_variant"]:
                sbert_table[col] = sbert_table[col].round(4)

        display(sbert_table.sort_values(by=f"NDCG@{K}", ascending=False))
    else:
        sbert_table = pd.DataFrame()
        print("Chưa có kết quả Sentence-BERT nào được tạo.")
else:
    sbert_table = pd.DataFrame()
    print("Bỏ qua Sentence-BERT vì chưa cài sentence-transformers.")

if len(sbert_errors) > 0:
    print("Một số cấu hình Sentence-BERT bị lỗi:")
    display(pd.DataFrame(sbert_errors))


Running Sentence-BERT config: {'model_name': 'all-MiniLM-L6-v2', 'text_variant': 'name_category_focus', 'batch_size': 64}


d:\DA2TEST\test\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 22/22 [00:44<00:00,  2.03s/it]


Running Sentence-BERT config: {'model_name': 'paraphrase-MiniLM-L6-v2', 'text_variant': 'name_category_about', 'batch_size': 64}


d:\DA2TEST\test\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--paraphrase-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 22/22 [00:23<00:00,  1.05s/it]


Running Sentence-BERT config: {'model_name': 'paraphrase-MiniLM-L6-v2', 'text_variant': 'category_name_about', 'batch_size': 64}


Batches: 100%|██████████| 22/22 [00:21<00:00,  1.03it/s]


,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10,sbert_model,text_variant,batch_size
0,"Sentence-BERT | model=all-MiniLM-L6-v2, text_v...",0.759,0.8236,0.99,4.0948,8.4865,all-MiniLM-L6-v2,name_category_focus,64
2,"Sentence-BERT | model=paraphrase-MiniLM-L6-v2,...",0.751,0.8154,0.98,4.0954,8.5204,paraphrase-MiniLM-L6-v2,category_name_about,64
1,"Sentence-BERT | model=paraphrase-MiniLM-L6-v2,...",0.726,0.7891,0.98,4.0972,8.5247,paraphrase-MiniLM-L6-v2,name_category_about,64


In [27]:

# ============================================================
# PHẦN 8: Lấy cấu hình Sentence-BERT tốt nhất
# ============================================================

if "sbert_table" in globals() and not sbert_table.empty:
    best_sbert = sbert_table.sort_values(
        by=f"NDCG@{K}",
        ascending=False
    ).head(1).copy()

    print("Best Sentence-BERT config:")
    display(best_sbert)
else:
    best_sbert = pd.DataFrame()
    print("Chưa có best_sbert vì Sentence-BERT chưa chạy hoặc chưa cài thư viện.")


Best Sentence-BERT config:


,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10,sbert_model,text_variant,batch_size
0,"Sentence-BERT | model=all-MiniLM-L6-v2, text_v...",0.759,0.8236,0.99,4.0948,8.4865,all-MiniLM-L6-v2,name_category_focus,64



## 6. Bảng so sánh cuối cùng

Bảng này chỉ so sánh các phương pháp thuộc nhóm **Content + Category**, đúng theo phạm vi bạn muốn:

1. **TF-IDF Content + Category**: baseline đang có trong notebook.
2. **TF-IDF Content + Category Tuned**: cấu hình TF-IDF tốt nhất sau tuning nếu bạn đã chạy phần tuning cũ.
3. **Weighted TF-IDF**: cấu hình tốt nhất sau tuning trọng số field.
4. **Sentence-BERT**: cấu hình tốt nhất sau tuning model/text variant nếu đã chạy được.

Chỉ số nên ưu tiên khi nhận xét: `NDCG@10`, vì chỉ số này không chỉ xem có recommend đúng category hay không, mà còn xét sản phẩm relevant có đứng ở vị trí cao trong top 10 hay không.


In [28]:

# ============================================================
# PHẦN 9: So sánh cuối cùng giữa các phương pháp Content + Category
# ============================================================

comparison_rows = []

# 1. Baseline TF-IDF Content + Category hiện tại
try:
    baseline_tfidf_cc = result_content_category.copy()
    baseline_tfidf_cc["Model"] = "TF-IDF Content + Category (current)"
except NameError:
    baseline_tfidf_cc = evaluate_model(
        "TF-IDF Content + Category (current)",
        recommend_content_with_category,
        k=K
    )
comparison_rows.append(baseline_tfidf_cc)

# 2. Best tuned TF-IDF Content + Category từ phần tuning cũ của bạn
if "tfidf_with_category_table" in globals() and not tfidf_with_category_table.empty:
    best_tuned_tfidf_cc = tfidf_with_category_table.sort_values(
        by=f"NDCG@{K}",
        ascending=False
    ).head(1).copy()
    best_tuned_tfidf_cc["Model"] = "TF-IDF Content + Category (best tuned)"
    comparison_rows.append(best_tuned_tfidf_cc)

# 3. Best Weighted TF-IDF
if "best_weighted_tfidf" in globals() and not best_weighted_tfidf.empty:
    best_weighted_row = best_weighted_tfidf.copy()
    best_weighted_row["Model"] = "Weighted TF-IDF (best tuned)"
    comparison_rows.append(best_weighted_row)

# 4. Best Sentence-BERT
if "best_sbert" in globals() and not best_sbert.empty:
    best_sbert_row = best_sbert.copy()
    best_sbert_row["Model"] = "Sentence-BERT (best tuned)"
    comparison_rows.append(best_sbert_row)

final_method_comparison = pd.concat(comparison_rows, ignore_index=True)

# Chỉ giữ các cột metric chính để bảng dễ nhìn.
metric_cols = [
    "Model",
    f"Precision@{K}",
    f"NDCG@{K}",
    f"HitRate@{K}",
    f"AvgRating@{K}",
    f"AvgPopularity@{K}"
]

available_metric_cols = [col for col in metric_cols if col in final_method_comparison.columns]
final_method_comparison = final_method_comparison[available_metric_cols]

for col in final_method_comparison.columns:
    if col != "Model":
        final_method_comparison[col] = final_method_comparison[col].round(4)

final_method_comparison.sort_values(by=f"NDCG@{K}", ascending=False)


,Model,Precision@10,NDCG@10,HitRate@10,AvgRating@10,AvgPopularity@10
2,Weighted TF-IDF (best tuned),0.8880,0.9611,0.9933,4.0786,8.3311
0,TF-IDF Content + Category (current),0.8137,0.8756,0.9900,4.0751,8.3523
1,TF-IDF Content + Category (best tuned),0.8137,0.8756,0.9900,4.0751,8.3523
3,Sentence-BERT (best tuned),0.7590,0.8236,0.9900,4.0948,8.4865



## 7. Gợi ý cách nhận xét kết quả trong báo cáo

Bạn có thể đọc bảng kết quả theo hướng sau:

- Nếu **Weighted TF-IDF** cao hơn TF-IDF Content + Category, có thể kết luận rằng việc tách field và gán trọng số cho `product_name`, `about_product`, `category` giúp hệ thống gợi ý bám sát sản phẩm gốc hơn.
- Nếu **Sentence-BERT** cao hơn TF-IDF, có thể giải thích rằng embedding ngữ nghĩa giúp nhận ra các sản phẩm tương tự ngay cả khi không trùng nhiều từ khóa.
- Nếu **TF-IDF hoặc Weighted TF-IDF** cao hơn Sentence-BERT, điều này vẫn hợp lý vì ground truth hiện tại đang dựa trên category. TF-IDF có category rõ ràng thường rất mạnh trong kiểu đánh giá này.
- `NDCG@10` nên được dùng làm chỉ số chính vì nó phản ánh cả độ đúng và vị trí xếp hạng trong top 10.
- `Precision@10` cho biết trong 10 sản phẩm gợi ý có bao nhiêu sản phẩm cùng category.
- `HitRate@10` cho biết hệ thống có ít nhất một gợi ý đúng category trong top 10 hay không.


In [29]:

# ============================================================
# PHẦN 10: Lưu kết quả ra CSV để đưa vào báo cáo
# ============================================================

if "weighted_tfidf_table" in globals() and not weighted_tfidf_table.empty:
    weighted_tfidf_table.to_csv("weighted_tfidf_tuning.csv", index=False)
    print("Saved: weighted_tfidf_tuning.csv")

if "sbert_table" in globals() and not sbert_table.empty:
    sbert_table.to_csv("sentence_bert_tuning.csv", index=False)
    print("Saved: sentence_bert_tuning.csv")

if "final_method_comparison" in globals() and not final_method_comparison.empty:
    final_method_comparison.to_csv("final_method_comparison.csv", index=False)
    print("Saved: final_method_comparison.csv")


Saved: weighted_tfidf_tuning.csv
Saved: sentence_bert_tuning.csv
Saved: final_method_comparison.csv



## 8. Tóm tắt ý nghĩa các phần code vừa thêm

| Phần | Nội dung | Mục đích |
|---|---|---|
| Phần 1 | Chuẩn bị `text_product_name`, `text_about_product`, `text_category` | Tách field để so sánh công bằng và phục vụ Weighted TF-IDF |
| Phần 2 | Xây hàm Weighted TF-IDF | Tính similarity riêng từng field rồi cộng theo trọng số |
| Phần 3 | Tuning Weighted TF-IDF | Tìm bộ `max_features`, `ngram_range`, `min_df`, `w_name`, `w_about`, `w_category` tốt nhất |
| Phần 4 | Chọn best Weighted TF-IDF | Lấy cấu hình có `NDCG@10` cao nhất |
| Phần 5 | Import Sentence-BERT | Kiểm tra thư viện đã cài chưa |
| Phần 6 | Xây hàm Sentence-BERT | Encode sản phẩm thành embedding và recommend bằng cosine similarity |
| Phần 7 | Tuning Sentence-BERT | So sánh model Sentence-BERT và cách ghép text đầu vào |
| Phần 8 | Chọn best Sentence-BERT | Lấy cấu hình Sentence-BERT có `NDCG@10` cao nhất |
| Phần 9 | Bảng so sánh cuối | So sánh TF-IDF Content + Category, Weighted TF-IDF và Sentence-BERT |
| Phần 10 | Lưu CSV | Xuất kết quả để đưa vào báo cáo |
